# Data Preprocessing

## Objetivo

Preparar los datos necesarios para construir el sistema de recomendación.

En esta etapa se integrarán las tablas principales, se seleccionarán las columnas relevantes y se generará una base procesada de interacciones usuario-producto.

In [1]:
# Importar librerías necesarias

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
# Cargar datasets principales

orders = pd.read_csv("../data/raw/instacart/orders.csv")
products = pd.read_csv("../data/raw/instacart/products.csv")
aisles = pd.read_csv("../data/raw/instacart/aisles.csv")
departments = pd.read_csv("../data/raw/instacart/departments.csv")
prior = pd.read_csv("../data/raw/instacart/order_products__prior.csv")

In [3]:
# Unir interacciones históricas con información de pedidos

interactions = prior.merge(
    orders,
    on="order_id",
    how="left"
)

interactions.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,202279,prior,3,5,9,8.0
1,2,28985,2,1,202279,prior,3,5,9,8.0
2,2,9327,3,0,202279,prior,3,5,9,8.0
3,2,45918,4,1,202279,prior,3,5,9,8.0
4,2,30035,5,0,202279,prior,3,5,9,8.0


In [4]:
# Enriquecer interacciones con información de productos

interactions = interactions.merge(
    products,
    on="product_id",
    how="left"
)

interactions.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13


In [5]:
# Enriquecer interacciones con subcategorías y categorías

interactions = interactions.merge(
    aisles,
    on="aisle_id",
    how="left"
)

interactions = interactions.merge(
    departments,
    on="department_id",
    how="left"
)

interactions.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [6]:
# Revisar dimensiones de la tabla consolidada

interactions.shape

(32434489, 15)

In [7]:
# Revisar uso de memoria

interactions.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 15 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   product_id              int64  
 2   add_to_cart_order       int64  
 3   reordered               int64  
 4   user_id                 int64  
 5   eval_set                object 
 6   order_number            int64  
 7   order_dow               int64  
 8   order_hour_of_day       int64  
 9   days_since_prior_order  float64
 10  product_name            object 
 11  aisle_id                int64  
 12  department_id           int64  
 13  aisle                   object 
 14  department              object 
dtypes: float64(1), int64(10), object(4)
memory usage: 10.2 GB


In [8]:
# Seleccionar columnas relevantes para modelado

cols_model = [
    "user_id",
    "product_id",
    "reordered",
    "product_name",
    "department",
    "aisle"
]

interactions_model = interactions[cols_model].copy()

In [9]:
# Revisar dimensiones de la nueva tabla

interactions_model.shape

(32434489, 6)

In [10]:
# Revisar memoria utilizada

interactions_model.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 6 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   user_id       int64 
 1   product_id    int64 
 2   reordered     int64 
 3   product_name  object
 4   department    object
 5   aisle         object
dtypes: int64(3), object(3)
memory usage: 6.6 GB


In [11]:
# Cantidad de interacciones por usuario

user_interactions = (
    interactions_model.groupby("user_id")
    .size()
    .reset_index(name="interaction_count")
)

user_interactions["interaction_count"].describe()

count    206209.000000
mean        157.289396
std         204.208233
min           3.000000
25%          39.000000
50%          83.000000
75%         188.000000
max        3725.000000
Name: interaction_count, dtype: float64

In [12]:
# Cantidad de interacciones por producto

product_interactions = (
    interactions_model.groupby("product_id")
    .size()
    .reset_index(name="interaction_count")
)

product_interactions["interaction_count"].describe()

count     49677.000000
mean        652.907563
std        4792.114416
min           1.000000
25%          17.000000
50%          60.000000
75%         260.000000
max      472565.000000
Name: interaction_count, dtype: float64

In [13]:
# Identificar productos con al menos 20 compras

valid_products = product_interactions[
    product_interactions["interaction_count"] >= 20
]["product_id"]

In [14]:
# Filtrar interacciones

interactions_filtered = interactions_model[
    interactions_model["product_id"].isin(valid_products)
].copy()

In [15]:
# Revisar tamaño del dataset filtrado

interactions_filtered.shape

(32300077, 6)

In [16]:
# Revisar cantidad de productos restantes

interactions_filtered["product_id"].nunique()

35922

In [17]:
# Revisar cantidad de usuarios restantes

interactions_filtered["user_id"].nunique()

206208

In [18]:
# Usuarios y productos del dataset filtrado

n_users = interactions_filtered["user_id"].nunique()
n_products = interactions_filtered["product_id"].nunique()

print(f"Usuarios: {n_users:,}")
print(f"Productos: {n_products:,}")
print(f"Posiciones teóricas: {n_users * n_products:,}")

Usuarios: 206,208
Productos: 35,922
Posiciones teóricas: 7,407,403,776


In [19]:
# Sparsity después del filtrado

n_interactions = len(interactions_filtered)

sparsity = 1 - (
    n_interactions /
    (n_users * n_products)
)

print(f"Sparsity: {sparsity:.4%}")

Sparsity: 99.5639%


# Conclusiones de Data Preprocessing

## Filtrado de productos

Se eliminaron productos con menos de 20 interacciones.

### Resultados

- Productos originales: 49.677
- Productos filtrados: 35.922

El filtro eliminó aproximadamente el 28% de los productos, pero menos del 1% de las interacciones totales.

Esto confirma la presencia de una distribución Long Tail, donde una gran cantidad de productos registra muy pocas compras.

## Sparsity

Luego del filtrado, la matriz usuario-producto mantiene una dispersión del 99,56%.

Este comportamiento es característico de sistemas de recomendación reales y justifica el uso de técnicas de filtrado colaborativo.

## Dataset para modelado

Se construyó una tabla consolidada de interacciones usuario-producto que servirá como base para las etapas de Feature Engineering y Modelado.

In [20]:
# Guardar dataset procesado

interactions_filtered.to_parquet(
    "../data/processed/interactions_filtered.parquet",
    index=False
)

In [21]:
# Verificar archivo guardado

import os

os.path.exists(
    "../data/processed/interactions_filtered.parquet"
)

True